# Player model: current level, weaknesses, confidence, departure from history

**Milestones:** "Estimate player habits from game history" and the fallback behaviour "when the
player model is uncertain, or the player departs sharply from their history, play solid
strength-limited moves" (README Teaching; "The player model can be wrong" row).

The model keeps one `GameSummary` per analysed game (built from `habits.MoveAnalysis`) and gives:

* **Level**: Elo estimate from ACPL with a confidence interval. The ACPL->Elo formula below is a
  *placeholder* (`TODO-calibrate`): ACPL depends heavily on the analyst engine, depth and time
  control, so it must be refit on games analysed by *our* analyst with known ratings.
* **Weaknesses**: blunder / mistake / missed-forcing-move rates with Wilson intervals, and ACPL per
  phase.
* **Confidence** in 0..1: how much data backs the estimate (placeholder formula, documented).
* **OOD score**: how far a new game is from the player's history (max |z| over features). High =
  the player is behaving unlike their past (new openings, much stronger/weaker, hiding strength).

**TODO checklist**
- [x] Summaries, estimate with interval, Wilson weakness intervals, OOD z-score, JSON save/load.
- [ ] TODO-calibrate `estimate_elo_from_acpl` (fit on rated games analysed by our analyst).
- [ ] Recency weighting: a returning player's old games should count less (README "rust").
- [ ] Weakness relative to expected rates *at the player's level* (needs reference data).
- [ ] Answer the README open question: "from how many games?" (min_games in game.yaml).

In [ ]:
%run ./habits.ipynb
%run ./metrics.ipynb

In [ ]:
import json
import math
import statistics
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional

CONF_CFG = TEACH_CFG["confidence"]

## Level from ACPL (placeholder, TODO-calibrate)

`elo = 3100 * exp(-0.01 * acpl)`, clamped to 400..3000. Exponential decay is a commonly used rough
shape (ACPL ~ 20 for top players, ~ 100+ for beginners), but the constants are **not** fitted.
The function is monotone decreasing, so an ACPL interval maps directly to an Elo interval.

In [ ]:
ELO_A, ELO_K = 3100.0, 0.01        # TODO-calibrate


def estimate_elo_from_acpl(acpl: float) -> float:
    return float(min(3000.0, max(400.0, ELO_A * math.exp(-ELO_K * acpl))))


def acpl_interval(per_game_acpl: list, z: float = 1.96, sd_floor: float = 15.0) -> tuple[float, float, float]:
    """(mean, low, high) of ACPL across games; sd floored so 1-2 games do not look certain."""
    n = len(per_game_acpl)
    if n == 0:
        return float("nan"), 0.0, float(TEACH_CFG["cpl_cap"])
    mean = statistics.fmean(per_game_acpl)
    sd = max(sd_floor, statistics.stdev(per_game_acpl) if n > 1 else sd_floor * 3)
    half = z * sd / math.sqrt(n)
    return mean, max(0.0, mean - half), mean + half

## Game summaries

In [ ]:
FEATURES = ("acpl", "blunder_rate", "mistake_rate", "missed_forcing_rate")


@dataclass
class GameSummary:
    game_id: str
    n_moves: int
    acpl: float
    counts: dict                      # classification -> count
    phase_cpl_sum: dict               # phase -> sum of CPL
    phase_moves: dict                 # phase -> number of moves
    n_forcing: int
    n_missed_forcing: int
    mean_think_time_s: Optional[float] = None

    @property
    def blunder_rate(self) -> float:
        return self.counts.get("blunder", 0) / max(1, self.n_moves)

    @property
    def mistake_rate(self) -> float:
        return self.counts.get("mistake", 0) / max(1, self.n_moves)

    @property
    def missed_forcing_rate(self) -> float:
        return self.n_missed_forcing / self.n_forcing if self.n_forcing else 0.0

    def feature(self, name: str) -> float:
        return float(getattr(self, name))


def summarize_game(analyses: list[MoveAnalysis], game_id: Optional[str] = None) -> GameSummary:
    if not analyses:
        raise ValueError("no moves to summarise")
    counts, psum, pn = {}, {}, {}
    for a in analyses:
        counts[a.classification] = counts.get(a.classification, 0) + 1
        psum[a.phase] = psum.get(a.phase, 0) + a.cpl
        pn[a.phase] = pn.get(a.phase, 0) + 1
    tts = [a.think_time_s for a in analyses if a.think_time_s is not None]
    return GameSummary(game_id or analyses[0].game_id, len(analyses),
                       statistics.fmean(a.cpl for a in analyses), counts, psum, pn,
                       sum(a.best_was_forcing for a in analyses), sum(a.missed_forcing for a in analyses),
                       statistics.fmean(tts) if tts else None)

## Estimates

In [ ]:
@dataclass
class Weakness:
    name: str
    rate: float
    low: float
    high: float
    n: int


@dataclass
class PlayerEstimate:
    elo: float
    elo_low: float
    elo_high: float
    acpl: float
    confidence: float
    n_games: int
    n_moves: int
    weaknesses: list
    acpl_by_phase: dict


class PlayerModel:
    def __init__(self, player: str, cfg: dict = CONF_CFG):
        self.player, self.cfg = player, cfg
        self.games: list[GameSummary] = []

    # -- updating ---------------------------------------------------------------
    def add_game_summary(self, s: GameSummary) -> None:
        self.games.append(s)

    def update(self, analyses: list[MoveAnalysis], game_id: Optional[str] = None) -> GameSummary:
        """Call after each game with the player's analysed moves."""
        s = summarize_game(analyses, game_id)
        self.add_game_summary(s)
        return s

    @property
    def n_moves(self) -> int:
        return sum(g.n_moves for g in self.games)

    # -- confidence -------------------------------------------------------------
    def confidence(self) -> float:
        """Placeholder: data term = min(1, games/min_games) * min(1, moves/min_moves), times a width
        term 1 / (1 + elo_ci_width / 800). TODO-calibrate with OOD tests (ood_testing.ipynb)."""
        if not self.games:
            return 0.0
        data = min(1.0, len(self.games) / self.cfg["min_games"]) * min(1.0, self.n_moves / self.cfg["min_moves"])
        _, lo, hi = acpl_interval([g.acpl for g in self.games])
        width = estimate_elo_from_acpl(lo) - estimate_elo_from_acpl(hi)
        return float(data / (1 + width / 800))

    # -- estimate ---------------------------------------------------------------
    def weaknesses(self) -> list[Weakness]:
        n = self.n_moves
        nb = sum(g.counts.get("blunder", 0) for g in self.games)
        nm = sum(g.counts.get("mistake", 0) for g in self.games)
        nf = sum(g.n_forcing for g in self.games)
        nmf = sum(g.n_missed_forcing for g in self.games)
        out = []
        for name, k, total in (("blunder", nb, n), ("mistake", nm, n), ("missed_forcing", nmf, nf)):
            lo, hi = wilson_interval(k, total)
            out.append(Weakness(name, k / total if total else 0.0, lo, hi, total))
        return sorted(out, key=lambda w: -w.low)          # most confidently frequent first

    def estimate(self) -> PlayerEstimate:
        mean, lo, hi = acpl_interval([g.acpl for g in self.games])
        psum, pn = {}, {}
        for g in self.games:
            for ph, v in g.phase_cpl_sum.items():
                psum[ph] = psum.get(ph, 0) + v
                pn[ph] = pn.get(ph, 0) + g.phase_moves[ph]
        elo = estimate_elo_from_acpl(mean) if self.games else float(ENGINE_CFG["default_elo"])
        return PlayerEstimate(elo, estimate_elo_from_acpl(hi), estimate_elo_from_acpl(lo), mean,
                              self.confidence(), len(self.games), self.n_moves, self.weaknesses(),
                              {ph: psum[ph] / pn[ph] for ph in psum})

    # -- departure from history -------------------------------------------------
    def ood_score(self, game: GameSummary | list, sd_floor: dict | None = None) -> float:
        """max |z| of the game's features vs the history. inf if fewer than 2 past games
        (no history = cannot judge = treat as out of distribution)."""
        if isinstance(game, list):
            game = summarize_game(game)
        if len(self.games) < 2:
            return math.inf
        floors = sd_floor or {"acpl": 10.0, "blunder_rate": 0.02, "mistake_rate": 0.02, "missed_forcing_rate": 0.1}
        z = []
        for f in FEATURES:
            hist = [g.feature(f) for g in self.games]
            sd = max(floors[f], statistics.stdev(hist))
            z.append(abs(game.feature(f) - statistics.fmean(hist)) / sd)
        return float(max(z))

    def is_ood(self, game) -> bool:
        return self.ood_score(game) > self.cfg["max_ood_score"]

    # -- persistence ------------------------------------------------------------
    def save(self, path: Path | str) -> None:
        Path(path).write_text(json.dumps({"player": self.player, "games": [asdict(g) for g in self.games]}, indent=1))

    @classmethod
    def load(cls, path: Path | str) -> "PlayerModel":
        d = json.loads(Path(path).read_text())
        m = cls(d["player"])
        m.games = [GameSummary(**g) for g in d["games"]]
        return m

## Self-check (synthetic summaries, no engine needed)

In [ ]:
import random
import tempfile

assert estimate_elo_from_acpl(0) == 3000 and estimate_elo_from_acpl(10_000) == 400
assert estimate_elo_from_acpl(40) > estimate_elo_from_acpl(80)


def _synthetic(gid, acpl, n=40, blunders=1, rng=random.Random(0)):
    return GameSummary(gid, n, acpl, {"blunder": blunders, "mistake": 2, "good": n - blunders - 2},
                       {"middlegame": acpl * n}, {"middlegame": n}, 8, 2)

m = PlayerModel("tester")
assert m.confidence() == 0.0 and math.isinf(m.ood_score(_synthetic("x", 50)))
for i in range(8):
    m.add_game_summary(_synthetic(f"g{i}", 55 + (i % 3) * 5))
est = m.estimate()
assert est.elo_low <= est.elo <= est.elo_high and 0.5 < est.confidence <= 1.0, est
assert est.n_games == 8 and est.weaknesses[0].low >= est.weaknesses[-1].low
assert m.ood_score(_synthetic("same", 60)) < CONF_CFG["max_ood_score"]
assert m.is_ood(_synthetic("much_weaker", 200, blunders=8))
few = PlayerModel("new"); few.add_game_summary(_synthetic("a", 60))
assert few.confidence() < est.confidence

# real analyses -> summary -> model
s = m.update(black, "scholar")
assert s.n_moves == 3 and s.counts["blunder"] == 1
with tempfile.TemporaryDirectory() as tmp:
    m.save(Path(tmp) / "p.json")
    m2 = PlayerModel.load(Path(tmp) / "p.json")
    assert len(m2.games) == 9 and m2.estimate().acpl == m.estimate().acpl
print("player_model self-check passed")